Identical split to training: seed 42, 90/10

In [ ]:
import json, random, math, os, gc
import numpy as np, pandas as pd, torch
 
DATASET_PATH = "tanglish.jsonl"
ADAPTER_DIR  = "tanglish_gemma2b_adapter"
BASE_ID      = "google/gemma-2b"
IT_ID        = "google/gemma-2b-it"
MAX_NEW      = 200
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
 
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]
 
def fmt(r):
    ins, inp, out = (r.get("instruction", "").strip(), r.get("input", "").strip(), r.get("output", "").strip())
    user = f"{ins}\n\nContext:\n{inp}" if inp else ins
    return {
        "text": f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>",
        "instruction": ins, "input": inp, "output": out,
    }
 
formatted = [fmt(r) for r in raw]
random.seed(42)
random.shuffle(formatted)
cut = int(len(formatted) * 0.9)
train, val = formatted[:cut], formatted[cut:]
print(f"Train {len(train)} | Val {len(val)}")
 
MODEL_TAG = "<start_of_turn>model\n"
def split_prompt_target(text):
    prompt = text.split(MODEL_TAG)[0] + MODEL_TAG
    target = text.split(MODEL_TAG)[-1].replace("<end_of_turn>", "").strip()
    return prompt, target
 
val_prompts, val_refs = zip(*[split_prompt_target(v["text"]) for v in val])
val_prompts, val_refs = list(val_prompts), list(val_refs)
